In [ ]:
# --- Imports & path setup -------------------------------------------------
import sys, json
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd
from IPython.display import Image, display

from data_loader import DEVICE, PROJECT_ROOT, get_dataset_info
from moo_pymoo import run_pymoo_study
from analyze_study import analyze

print(f"Device       : {DEVICE}")
print(f"Project root : {PROJECT_ROOT}")
if str(DEVICE) == "cuda":
    import torch
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# --- Configuration --------------------------------------------------------
DATASET           = "fashion_mnist"
POP_SIZE          = 40
N_GEN             = 5
TRAIN_SUBSET_SIZE = None        # None = full training set
USE_AMP           = True        # fp16 mixed precision (CUDA only)
INFERENCE_WARMUP  = 10
INFERENCE_TIMED   = 100
NUM_WORKERS       = 4
SEED              = 42

COMPARE_PARETO    = None        # set to a botorch pareto_front.csv for joint-front overlay

info = get_dataset_info(DATASET)
total_trials = POP_SIZE * N_GEN
print(f"Dataset        : {DATASET}  (classes={info['num_classes']}, native res={info['default_resolution']})")
print(f"Total trials   : {total_trials} (pop={POP_SIZE} × gen={N_GEN})")
print(f"Train subset   : {'FULL DATASET' if TRAIN_SUBSET_SIZE is None else f'{TRAIN_SUBSET_SIZE:,} samples'}")
print(f"AMP            : {USE_AMP and str(DEVICE)=='cuda'}")

In [ ]:
summary, study_dir = run_pymoo_study(
    dataset_name=DATASET,
    pop_size=POP_SIZE,
    n_gen=N_GEN,
    train_subset_size=TRAIN_SUBSET_SIZE,
    seed=SEED,
    num_workers=NUM_WORKERS,
    use_amp=USE_AMP,
    inference_warmup=INFERENCE_WARMUP,
    inference_timed=INFERENCE_TIMED,
)

print("\n" + "=" * 70)
print(f"Study dir : {study_dir}")
print(f"Wall time : {summary['elapsed_seconds']/60:.1f} min")
print(f"Pareto    : {summary['n_pareto_points']} non-dominated points (final population)")

In [ ]:
compare_path = Path(COMPARE_PARETO) if COMPARE_PARETO else None
metrics = analyze(study_dir, compare_pareto_path=compare_path)

print(f"n_trials       : {metrics['n_trials']}")
print(f"n_pareto       : {metrics['n_pareto_points']}")
print(f"hypervolume    : {metrics['hypervolume']:,.2f}")
print(f"spacing        : {metrics['spacing']:.4f}")
print(f"acc max        : {metrics['extremes']['accuracy_max']:.4f}")
print(f"ms  min        : {metrics['extremes']['inference_ms_min']:.4f}")
print(f"params min     : {metrics['extremes']['param_count_min']:,}")
if 'comparison' in metrics:
    c = metrics['comparison']
    print("\n--- vs comparison front ---")
    print(f"joint pareto      : {c['joint_n_pareto']}")
    print(f"ours surviving    : {c['ours_surviving_in_joint']}")
    print(f"other surviving   : {c['other_surviving_in_joint']}")
    print(f"GD ours -> joint  : {c['gd_ours_to_joint']:.6f}")
    print(f"GD other-> joint  : {c['gd_other_to_joint']:.6f}")

In [ ]:
pareto_table = pd.read_csv(study_dir / "pareto_table.csv")
pareto_table

In [ ]:
display(Image(filename=str(study_dir / "plot_2d_panels.png")))

In [ ]:
display(Image(filename=str(study_dir / "plot_parallel_coords.png")))

In [ ]:
display(Image(filename=str(study_dir / "plot_3d_scatter.png")))

In [ ]:
display(Image(filename=str(study_dir / "plot_3d_pareto.png")))

In [ ]:
with open(study_dir / "appendix_solution.json") as f:
    appendix = json.load(f)

print(f"Label              : {appendix['label']}")
print(f"Trial number       : {appendix.get('trial_number')}")
print()
print("--- Objectives ---")
for k, v in appendix['objectives'].items():
    print(f"  {k:<14s} : {v}")
print()
print("--- Decision variables ---")
for k, v in appendix['decision_variables'].items():
    print(f"  {k:<18s} : {v}")

In [ ]:
candidate = PROJECT_ROOT / "results" / "botorch" / DATASET
if candidate.exists():
    runs = sorted([p for p in candidate.iterdir() if p.is_dir()])
    if runs:
        latest = runs[-1] / "pareto_front.csv"
        print(f"botorch latest pareto: {latest}")
        print(f"\nTo enable comparison, set in the config cell:\n  COMPARE_PARETO = r\"{latest}\"")
    else:
        print(f"(no botorch runs yet in {candidate})")
else:
    print(f"(no botorch results for dataset={DATASET})")

In [ ]:
HF_DATASET_REPO = None    # e.g. "kvelidanda/moml-results"  -- set to enable upload

if HF_DATASET_REPO:
    import os
    from huggingface_hub import HfApi, login

    if "HF_TOKEN" in os.environ:
        login(token=os.environ["HF_TOKEN"])

    api = HfApi()
    api.create_repo(repo_id=HF_DATASET_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(
        folder_path=str(study_dir),
        path_in_repo=f"pymoo/{study_dir.name}",
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        commit_message=f"pymoo NSGA-II study: {study_dir.name}",
    )
    print(f"Uploaded → https://huggingface.co/datasets/{HF_DATASET_REPO}/tree/main/pymoo/{study_dir.name}")
else:
    print("Upload skipped. Set HF_DATASET_REPO above to enable.")